# Exceptions => Context Managers

A **context manager** sets something up, runs your block, and then **always cleans up**, even if an exception happens. You use it with the `with` statement.

| Tool | Purpose |
|---|---|
| `with x as y:` | Run a block inside a context manager |
| `__enter__(self)` | Runs at the start. Its return value goes to `as` |
| `__exit__(self, exc_type, exc, tb)` | Runs at the end. Return `True` to suppress the exception |
| `@contextmanager` | Build a context manager from a generator function |
| `contextlib.suppress(...)` | Ignore the listed exceptions |
| `contextlib.redirect_stdout(f)` | Send `print()` output to another file |
| `contextlib.closing(obj)` | Call `obj.close()` at the end |
| `contextlib.ExitStack` | Manage a variable number of context managers |

---

## Why `with`?

`with` replaces a `try / finally`:

```python
# Manual cleanup
file = open("data.txt")
try:
    data = file.read()
finally:
    file.close()

# Same thing with `with`
with open("data.txt") as file:
    data = file.read()
```

Typical uses: files, locks, database transactions, temporary changes, timers.

---

## Class-Based Context Manager

```python
class Timer:
    def __enter__(self):
        # set up
        return self

    def __exit__(self, exc_type, exc, tb):
        # clean up
        return False
```

| `__exit__` argument | Meaning |
|---|---|
| `exc_type`, `exc`, `tb` | The exception class, instance and traceback. All `None` when no error happened |
| Return value | `True` swallows the exception. `False` (or `None`) lets it continue |

### Important

* `__exit__` runs **even when the block raises**.
* Suppressing exceptions by returning `True` hides bugs. Do it rarely and on purpose.

---

## Generator-Based Context Manager

`@contextmanager` turns a generator with **one `yield`** into a context manager.

```python
from contextlib import contextmanager

@contextmanager
def opened(name):
    print("setup")
    try:
        yield name          # the block runs here
    finally:
        print("cleanup")    # always runs
```

* Code before `yield` is the setup.
* The yielded value goes to `as`.
* Put the cleanup in `finally`, so it runs even if the block fails.

---

## Multiple Context Managers

```python
with open("a.txt") as a, open("b.txt") as b:
    ...

with (
    open("a.txt") as a,
    open("b.txt") as b,
):
    ...
```

The parenthesized form works in Python 3.10 and later.

---

## Useful Tools in `contextlib`

| Tool | Example |
|---|---|
| `suppress` | `with suppress(FileNotFoundError): os.remove(path)` |
| `redirect_stdout` | `with redirect_stdout(buffer): print("captured")` |
| `closing` | `with closing(resource): ...` |
| `ExitStack` | Open a list of files in one `with` |

## Source

https://docs.python.org/3/reference/compound_stmts.html#the-with-statement

https://docs.python.org/3/library/contextlib.html

In [ ]:
import io
import os
import tempfile
import time
from contextlib import ExitStack, contextmanager, redirect_stdout, suppress

# Class-based: __enter__ / __exit__
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.start
        return False                                # do not swallow exceptions

with Timer() as timer:
    sum(range(10_000))
print(timer.elapsed >= 0)

# __exit__ runs even when the block raises
class Resource:
    def __enter__(self):
        print("open")
        return self

    def __exit__(self, exc_type, exc, tb):
        print("close, error was:", exc_type.__name__ if exc_type else None)
        return False

try:
    with Resource():
        raise ValueError("boom")
except ValueError as error:
    print("caught outside:", error)

# Returning True suppresses the exception
class Quiet:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        return True

with Quiet():
    1 / 0
print("execution continues after the with block")

# Generator-based: @contextmanager
@contextmanager
def opened(name):
    print("setup", name)
    try:
        yield name.upper()
    finally:
        print("cleanup", name)

with opened("db") as value:
    print("inside", value)

try:
    with opened("db2"):
        raise RuntimeError("failed")
except RuntimeError:
    print("cleanup still ran")

# A temporary change that is always undone
@contextmanager
def changed_directory(path):
    previous = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)

start = os.getcwd()
with tempfile.TemporaryDirectory() as tmp:
    with changed_directory(tmp):
        print(os.path.samefile(os.getcwd(), tmp))
print(os.getcwd() == start)

# contextlib helpers
with suppress(ZeroDivisionError):
    1 / 0
print("suppressed")

buffer = io.StringIO()
with redirect_stdout(buffer):
    print("captured text")
print(repr(buffer.getvalue()))

# Multiple context managers in one statement
with opened("a") as first, opened("b") as second:
    print(first, second)

# ExitStack: a variable number of context managers
with ExitStack() as stack:
    managers = [stack.enter_context(opened(name)) for name in ("x", "y")]
    print(managers)